# Moirai variants & leakage-free ensembles (h = 6)

Upstream source: `Morai_improv2.ipynb` in the thesis workbench. Renamed from the early-plan name `04_moirai_fourier_sweep.ipynb` to the more accurate `04_moirai_variants_ensemble.ipynb` after inspecting the cell structure.

**Input**: `DTW_DFS_DIR / 'dtw_20_dfs' / *.parquet`.
**Output**: `moirai_results_dir('dtw20')` - multiple per-variant CSVs (`moirai_fourier_small_ensemble_h6_safe.csv`, `moirai_fourier_small_bestK3_h6.csv`, and K-sweep outputs under `moirai_results_dtw20_fourier_sweep_h6/K{3,4,5}_std1_ctx75/`).

**Variants included** (one experiment block per cell):
1. Moirai-1.1-R-small H=6 improvement run: calendar (future) + lags/rolls (past-only) + multi-context ensemble.
2. Moirai-1.1-R-large H=6 auto-adaptive, A100-optimised.
3. Neighbor-lag ensemble re-run for fallback-only keywords, best-per-keyword selection.
4. H=6++ batched, fixed-dim features, 3-context ensemble, AMP, compile.
5. Leakage-free evaluation: train/val/test split, method selection via validation only.
6. Leakage-free meta-ensemble: validation-derived weights, honest test.
7. Leakage-free per-series context selection with gated blend.


## Canonical cell for the paper

The paper cites this notebook for the **leakage-free robustness check**. The canonical cell is titled *FIXED: Leakage-free meta-ensemble (no-NaN), validation-derived weights, honest test*: ensemble weights are fit on validation only and applied to an honest held-out test split. The notebook is not a headline-result producer; the earlier cells sweep model scales (small vs large, A100-optimised, batched 3-context ensembles) and the last cell runs per-series context selection with a gated blend, supporting the meta-ensemble's robustness narrative.


In [ ]:
# --- Paper-repo bootstrap (replaces original Colab `drive.mount` cell) ---
import sys, os
from pathlib import Path

_REPO_SHARED = Path('..', '_shared').resolve()
if str(_REPO_SHARED) not in sys.path:
    sys.path.insert(0, str(_REPO_SHARED))

from paths import (  # noqa: E402
    ensure_env, DATA_ROOT, DTW_DFS_DIR, KEYWORDS_DIR_20, moirai_results_dir,
)
from api_keys import get_hf_token  # noqa: E402

ensure_env()
_hf_token = get_hf_token()
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token


In [ ]:
# --- Pin a stable stack for Py3.11 + uni2ts forecast path ---
!pip -q uninstall -y numpy pandas scipy transformers tokenizers huggingface-hub lightning torchmetrics pillow

# Core ABI pair (NumPy 1.26 + SciPy 1.11) + pandas
!pip -q install --no-deps numpy==1.26.4 pandas==2.2.2 scipy==1.11.4 pillow==11.0.0

# HF/Lightning stack that previously clashed
!pip -q install --no-deps transformers==4.46.2 tokenizers==0.22.0 huggingface-hub==0.36.0 lightning==2.3.3 torchmetrics==1.3.2

# (optional) make sure these are present
!pip -q install uni2ts==2.0.0 gluonts==0.14.3 polars==1.32.1 tqdm==4.66.5 pyarrow==22.0.0

# ---- Force runtime restart so NumPy/Pandas C-extensions reload cleanly ----
import os, signal, sys
os.kill(os.getpid(), 9)


In [ ]:
# Fix the HF mismatch: keep your current transformers, downgrade tokenizers
!pip -q uninstall -y tokenizers
!pip -q install --no-deps tokenizers==0.20.3

# sanity check
import tokenizers, transformers
print("tokenizers:", tokenizers.__version__)
print("transformers:", transformers.__version__)


In [ ]:
# ==== H=6 improvement run | Moirai-1.1-R-small | calendar (future) + lags/rolls (past-only) + multi-context ensemble ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ---------------- Params ----------------
FREQ         = "W"
H            = 6
CTX_CHOICES  = [48, 64, 80]     # ensemble contexts
NUM_SAMPLES  = 200              # MC samples per context
BATCH_SIZE   = 96               # A100 can handle this with 'small'
TRIM_PCT     = 0.10             # trimmed mean over samples (robust)
LAG_K        = 12               # lags 1..12
ROLL_MEANS   = [4, 12, 26]      # rolling means
ROLL_STDS    = [12]             # rolling stds

# ---------------- Device ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device, "| GPU:", torch.cuda.get_device_name(0) if device=='cuda' else "CPU")

# ---------------- Paths ----------------
IN_DIR  = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR = moirai_results_dir("dtw20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_H6_small_calendar_pastfeat_ensemble.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# ---------------- Helpers ----------------
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df = _read_pl_any(p)
    if "week" not in df.columns or "cpc_week" not in df.columns:
        return None
    base = (
        df.select(pl.col("week"),
                  pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
          .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
          .drop("week")
          .sort("ds")
          .filter(pl.col("y").is_not_null())
    ).to_pandas()
    base["ds"] = pd.to_datetime(base["ds"])
    base["y"]  = pd.to_numeric(base["y"], errors="coerce").astype(np.float32)
    base = base.dropna(subset=["y"]).sort_values("ds").reset_index(drop=True)
    # need enough history for features + context
    if len(base) < (max(CTX_CHOICES) + H + max(ROLL_MEANS+[1]) + 5):
        return None
    return base

def add_calendar(pdf: pd.DataFrame, k=3) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    return pdf

def add_past_feats(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.copy()
    # lags
    for k in range(1, LAG_K+1):
        pdf[f"y_lag_{k}"] = pdf["y"].shift(k)
    # rolling means/stds (past-only)
    for w in ROLL_MEANS:
        pdf[f"y_rollmean_{w}"] = pdf["y"].rolling(w, min_periods=1).mean().shift(1)
    for w in ROLL_STDS:
        pdf[f"y_rollstd_{w}"]  = pdf["y"].rolling(w, min_periods=2).std().shift(1)
    # fill early NaNs with expanding stats to avoid leakage
    for c in [c for c in pdf.columns if c.startswith("y_lag_") or c.startswith("y_roll")]:
        s = pdf[c]
        if s.isna().any():
            s2 = s.copy()
            # back/forward fill within the *past* (safe because we shift)
            s2 = s2.ffill().bfill()
            pdf[c] = s2.astype(np.float32)
        else:
            pdf[c] = s.astype(np.float32)
    return pdf

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def smape(y_true, y_pred):
    yt = np.asarray(y_true, np.float32)
    yp = np.asarray(y_pred, np.float32)
    denom = (np.abs(yt)+np.abs(yp))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(yp[m]-yt[m])/denom[m]))

def rmse(y_true, y_pred):
    yt = np.asarray(y_true, np.float32)
    yp = np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((yp-yt)**2)))

def trimmed_mean(arr, trim=0.1, axis=None):
    arr = np.sort(arr, axis=axis)
    n = arr.shape[axis] if axis is not None else arr.size
    k = int(n*trim)
    if axis is None:
        return arr[k:n-k].mean(dtype=np.float64)
    sl = [slice(None)]*arr.ndim
    sl[axis] = slice(k, n-k)
    return arr[tuple(sl)].mean(axis=axis, dtype=np.float64)

# ---------------- Model ----------------
module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def forecast_one(pdf: pd.DataFrame):
    # transforms + features
    pdf = add_calendar(add_past_feats(pdf))
    tr, te = split_train_test(pdf, H)
    if tr is None: return np.nan, np.nan

    # log transform (stabilize), fit on train only
    eps = 1e-6
    tr_y = np.log1p(np.maximum(tr["y"].values, 0) + eps).astype(np.float32)
    mu, sd = float(tr_y.mean()), float(tr_y.std() + 1e-8)
    pdf["y_z"] = (np.log1p(np.maximum(pdf["y"].values, 0) + eps).astype(np.float32) - mu)/sd

    fut_cols  = [c for c in pdf.columns if c.startswith("cal_")]  # future-known
    past_cols = [c for c in pdf.columns if c.startswith("y_lag_") or c.startswith("y_roll")]
    # windows for dataset
    tr_b = pdf.iloc[:len(tr)].copy()
    te_b = pdf.iloc[len(tr):].copy()

    # build base entry (we'll reuse arrays for different ctx)
    entry_base = {
        "start": pd.Timestamp(tr_b["ds"].iloc[0]),
        "target": tr_b["y_z"].to_numpy(np.float32),
    }
    if fut_cols:
        Xp = np.stack([tr_b[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        Xf = np.stack([te_b[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        entry_base["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)
    if past_cols:
        Xp_only = np.stack([tr_b[c].to_numpy(np.float32) for c in past_cols], axis=0)
        entry_base["past_feat_dynamic_real"] = Xp_only

    preds = []
    for ctx in CTX_CHOICES:
        forecaster = MoiraiForecast(
            prediction_length=H,
            target_dim=1,
            feat_dynamic_real_dim=len(fut_cols),
            past_feat_dynamic_real_dim=len(past_cols),
            context_length=ctx,
            module=module,
            patch_size="auto",
            num_samples=NUM_SAMPLES,
        )
        predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
        ds = ListDataset([entry_base], freq=FREQ)
        fc = next(predictor.predict(ds))

        # samples: (num_samples, H) -> robust aggregate along samples
        samples = np.asarray(fc.samples, dtype=np.float32)   # shape [S, H]
        zhat = trimmed_mean(samples, trim=TRIM_PCT, axis=0).astype(np.float32)
        preds.append(zhat)

    # ensemble across contexts
    zhat = np.vstack(preds).mean(axis=0).astype(np.float32)

    # invert transforms
    yhat = np.expm1(zhat*sd + mu) - eps
    yhat = np.maximum(yhat, 0).astype(np.float32)

    y = te["y"].to_numpy(np.float32)
    return smape(y, yhat), rmse(y, yhat)

# ---------------- Run ----------------
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")
rows = []
for p in tqdm(files, desc="H=6 | small + past-only feats + ensemble"):
    pdf = prep_keyword_pdf(p)
    if pdf is None:
        continue
    try:
        s, r = forecast_one(pdf)
    except Exception as e:
        print(f"[WARN] {p.stem} failed: {e}")
        s, r = np.nan, np.nan
    rows.append({
        "keyword": p.stem,
        "exog_setting": "H6_small_calendar+past_ens",
        "horizon_weeks": H,
        "smape_canonical": s,
        "rmse": r
    })

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print("✅ Done. Metrics:", OUT_CSV)

valid = out.dropna(subset=["smape_canonical","rmse"])
if not valid.empty:
    print(valid.agg(count=("rmse","count"),
                    sMAPE_mean=("smape_canonical","mean"),
                    sMAPE_median=("smape_canonical","median"),
                    RMSE_mean=("rmse","mean"),
                    RMSE_median=("rmse","median")))
else:
    print("No valid rows.")


In [ ]:
# ==== H=6 auto-adaptive | Moirai-1.1-R-large | optimized for A100 ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ---------------- Config ----------------
FREQ         = "W"
H            = 6
NUM_SAMPLES  = 160
BATCH_SIZE   = 32        # safe for A100 large model
CTX_MIN, CTX_MAX = 40, 96

# ---------------- Device ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device, "| GPU:", torch.cuda.get_device_name(0) if device=="cuda" else "CPU")

# ---------------- Paths ----------------
IN_DIR  = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR = moirai_results_dir("dtw20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_H6_auto_adaptive_large.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# ---------------- Helpers ----------------
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df = _read_pl_any(p)
    if "week" not in df.columns or "cpc_week" not in df.columns:
        return None
    base = (
        df.select(pl.col("week"),
                  pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
          .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
          .drop("week")
          .sort("ds")
          .filter(pl.col("y").is_not_null())
    ).to_pandas()
    base["ds"] = pd.to_datetime(base["ds"])
    base["y"]  = pd.to_numeric(base["y"], errors="coerce").astype(np.float32)
    base = base.dropna(subset=["y"]).sort_values("ds").reset_index(drop=True)
    if len(base) < (H + CTX_MIN + 5):
        return None
    return base

def add_calendar(pdf: pd.DataFrame, k=3) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    return pdf

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, np.float32), np.asarray(y_pred, np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred))/2.0
    m = denom != 0
    return float(100*np.mean(np.abs(y_pred[m]-y_true[m])/denom[m]))

def rmse(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, np.float32), np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((y_pred - y_true)**2)))

# ---------------- Model ----------------
print("Loading Moirai-1.1-R-large …")
module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-large").to(device).eval()
print("✅ Model loaded")

def run_series(pdf: pd.DataFrame):
    pdf = add_calendar(pdf)
    tr, te = split_train_test(pdf, H)
    if tr is None: return np.nan, np.nan

    # log1p normalization (fit on train)
    eps = 1e-6
    tr_y = np.log1p(np.maximum(tr["y"].values, 0) + eps).astype(np.float32)
    mu, sd = float(tr_y.mean()), float(tr_y.std() + 1e-8)
    pdf["y_z"] = (np.log1p(np.maximum(pdf["y"].values, 0) + eps).astype(np.float32) - mu)/sd

    fut_cols = [c for c in pdf.columns if c.startswith("cal_")]
    tr_b, te_b = pdf.iloc[:len(tr)].copy(), pdf.iloc[len(tr):].copy()

    entry = {"start": pd.Timestamp(tr_b["ds"].iloc[0]),
             "target": tr_b["y_z"].to_numpy(np.float32)}
    if fut_cols:
        Xp = np.stack([tr_b[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        Xf = np.stack([te_b[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)

    ctx = int(np.clip(round(len(tr)*0.4), CTX_MIN, CTX_MAX))
    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(fut_cols),
        past_feat_dynamic_real_dim=0,
        context_length=ctx,
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    ds = ListDataset([entry], freq=FREQ)
    fc = next(predictor.predict(ds))

    yhat = np.expm1(np.asarray(fc.mean, dtype=np.float32)*sd + mu) - eps
    yhat = np.maximum(yhat, 0).astype(np.float32)
    y = te["y"].to_numpy(np.float32)
    return smape(y, yhat), rmse(y, yhat)

# ---------------- Run all files ----------------
rows = []
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")
for p in tqdm(files, desc="H=6 | Moirai-large"):
    pdf = prep_keyword_pdf(p)
    if pdf is None:
        continue
    try:
        s, r = run_series(pdf)
    except Exception as e:
        print(f"[WARN] {p.stem} failed: {e}")
        s, r = np.nan, np.nan
    rows.append({
        "keyword": p.stem,
        "exog_setting": "H6_auto_large",
        "horizon_weeks": H,
        "smape_canonical": s,
        "rmse": r
    })

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print("✅ Done. Metrics:", OUT_CSV)

valid = out.dropna(subset=["smape_canonical","rmse"])
if not valid.empty:
    print(valid.agg(count=("rmse","count"),
                    sMAPE_mean=("smape_canonical","mean"),
                    sMAPE_median=("smape_canonical","median"),
                    RMSE_mean=("rmse","mean"),
                    RMSE_median=("rmse","median")))
else:
    print("No valid rows.")


In [ ]:
# === Re-run advanced neighbor-lag ensemble ONLY for fallback-only keywords, then pick best per keyword ===
from pathlib import Path
import numpy as np, pandas as pd, polars as pl
from datetime import date
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ---- Config (keeps the H=6 setup that got ~30.49 sMAPE on T4) ----
FREQ        = "W"
H           = 6
NUM_SAMPLES = 160
BATCH_SIZE  = 64
CTX_MIN, CTX_MAX = 48, 64     # adaptive, short contexts for stability

IN_DIR   = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR  = moirai_results_dir("dtw20")
METRICS  = OUT_DIR / "moirai_H6_t4_neighborlag_ensemble.csv"   # your combined file (advanced + fallback)
BEST_OUT = OUT_DIR / "moirai_H6_best_per_keyword.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# ---- IO helpers ----
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_pdf(p: Path) -> pd.DataFrame | None:
    df = _read_pl_any(p)
    if "week" not in df.columns or "cpc_week" not in df.columns:
        return None
    pdf = (
        df.select(pl.col("week"),
                  pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"),
                  *[c for c in df.columns if c not in {"week","cpc_week"}])
          .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
          .drop("week")
          .sort("ds")
    ).to_pandas()
    pdf["ds"] = pd.to_datetime(pdf["ds"])
    pdf["y"]  = pd.to_numeric(pdf["y"], errors="coerce").astype(np.float32)
    pdf = pdf.dropna(subset=["y"]).sort_values("ds").reset_index(drop=True)
    return pdf

def add_calendar(pdf: pd.DataFrame, k=3) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    return pdf

def add_target_lags(pdf: pd.DataFrame, max_lag=12, rolls=(4,12,26)) -> pd.DataFrame:
    pdf = pdf.copy()
    for k in range(1, max_lag+1):
        pdf[f"y_lag_{k}"] = pdf["y"].shift(k)
    for w in rolls:
        pdf[f"y_rollmean_{w}"] = pdf["y"].rolling(w, min_periods=1).mean().shift(1)
    lag_cols = [c for c in pdf.columns if c.startswith(("y_lag_","y_rollmean_"))]
    pdf[lag_cols] = pdf[lag_cols].ffill().bfill().astype(np.float32)
    return pdf

def detect_neighbor_cols(pdf: pd.DataFrame) -> list[str]:
    # Any numeric CPC-like cols except the main 'y'
    cand = [c for c in pdf.columns if "cpc" in c.lower() and c.lower() != "cpc_week"]
    # keep numeric only
    keep = []
    for c in cand:
        s = pd.to_numeric(pdf[c], errors="coerce")
        if s.notna().sum() >= 16:
            pdf[c] = s.astype(np.float32)
            keep.append(c)
    return keep[:20]  # cap to 20 for stability

def build_neighbor_feats(pdf: pd.DataFrame, neigh_cols: list[str]) -> pd.DataFrame:
    pdf = pdf.copy()
    if not neigh_cols:
        return pdf
    # robust fill
    pdf[neigh_cols] = pdf[neigh_cols].ffill().bfill()
    # neighbor aggregates + simple lags
    pdf["nb_mean"] = pdf[neigh_cols].mean(axis=1).astype(np.float32)
    for k in (1,2,3):
        pdf[f"nb_mean_lag_{k}"] = pdf["nb_mean"].shift(k)
    fill_cols = ["nb_mean"] + [f"nb_mean_lag_{k}" for k in (1,2,3)]
    pdf[fill_cols] = pdf[fill_cols].ffill().bfill().astype(np.float32)
    return pdf

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def smape(y_true, y_pred):
    yt, yp = np.asarray(y_true, np.float32), np.asarray(y_pred, np.float32)
    denom = (np.abs(yt)+np.abs(yp))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(yp[m]-yt[m])/denom[m]))

def rmse(y_true, y_pred):
    yt, yp = np.asarray(y_true, np.float32), np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((yp-yt)**2)))

# ---- Load Moirai (small) ----
module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def adaptive_ctx(n_train: int) -> int:
    # keep within [CTX_MIN, CTX_MAX], but not more than 1/2 history
    return int(np.clip(max(32, n_train//2), CTX_MIN, CTX_MAX))

def run_advanced(pdf: pd.DataFrame):
    pdf = add_calendar(pdf)
    pdf = add_target_lags(pdf, max_lag=12, rolls=(4,12,26))
    neigh_cols = detect_neighbor_cols(pdf)
    pdf = build_neighbor_feats(pdf, neigh_cols)

    tr, te = split_train_test(pdf, H)
    if tr is None: return np.nan, np.nan

    # log-stable scaling on train
    eps = 1e-6
    tr_log = np.log1p(np.maximum(tr["y"].values, 0) + eps).astype(np.float32)
    med = float(np.median(tr_log))
    iqr = float(np.percentile(tr_log, 75) - np.percentile(tr_log, 25) + 1e-6)

    def z(y): return (np.log1p(np.maximum(y, 0)+eps).astype(np.float32) - med)/iqr
    pdf["y_z"] = z(pdf["y"].values)

    fut_cols  = [c for c in pdf.columns if c.startswith(("cal_sin_","cal_cos_","cal_trend"))]
    past_cols = [c for c in pdf.columns if c.startswith(("y_lag_","y_rollmean_","nb_mean_lag_"))]

    # build GluonTS entry
    tr_b, te_b = pdf.iloc[:len(tr)].copy(), pdf.iloc[len(tr):].copy()
    entry = {"start": pd.Timestamp(tr_b["ds"].iloc[0]), "target": tr_b["y_z"].to_numpy(np.float32)}
    if fut_cols:
        Xp = np.stack([tr_b[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        Xf = np.stack([te_b[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)
    if past_cols:
        Xp_only = np.stack([tr_b[c].to_numpy(np.float32) for c in past_cols], axis=0)
        entry["past_feat_dynamic_real"] = Xp_only

    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(fut_cols),
        past_feat_dynamic_real_dim=len(past_cols),
        context_length=adaptive_ctx(len(tr)),
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    ds = ListDataset([entry], freq=FREQ)
    fc = next(predictor.predict(ds))
    zhat = np.asarray(fc.mean, dtype=np.float32)
    yhat_m = np.expm1(zhat*iqr + med) - 1e-6
    yhat_m = np.maximum(yhat_m, 0).astype(np.float32)

    # simple ensemble with naive components
    last_val = np.repeat(tr["y"].iloc[-1], H).astype(np.float32)
    if "nb_mean" in pdf.columns:
        nb_last = np.repeat(tr["nb_mean"].iloc[-1], H).astype(np.float32)
        yhat = 0.70*yhat_m + 0.15*last_val + 0.15*nb_last
    else:
        yhat = 0.80*yhat_m + 0.20*last_val

    y = te["y"].to_numpy(np.float32)
    return smape(y, yhat), rmse(y, yhat)

# ---- Figure out which keywords are fallback-only, re-run them, append rows ----
all_rows = pd.read_csv(METRICS)
fallback_only = (all_rows["exog_setting"] == "H6_T4_fallback")
fallback_kw = set(all_rows.loc[fallback_only, "keyword"])
advanced_kw = set(all_rows.loc[~fallback_only, "keyword"])
todo_kw = sorted(list(fallback_kw - advanced_kw))
print(f"Re-running advanced on fallback-only keywords: {len(todo_kw)}")

rows_new = []
for kw in tqdm(todo_kw, desc="Advanced retry"):
    p = next((pp for pp in IN_DIR.glob("*") if pp.stem == kw), None)
    if p is None: continue
    pdf = prep_pdf(p)
    if pdf is None or len(pdf) < 20: continue
    try:
        s, r = run_advanced(pdf)
        if not (np.isnan(s) or np.isnan(r)):
            rows_new.append({
                "keyword": kw,
                "exog_setting": "H6_T4_nbmean_retry",
                "horizon_weeks": 6,
                "smape_canonical": s,
                "rmse": r
            })
    except Exception as e:
        # keep quiet; we still have the fallback row
        pass

if rows_new:
    pd.DataFrame(rows_new).to_csv(METRICS, mode="a", index=False, header=False)
    print(f"✅ Appended {len(rows_new)} advanced retries → {METRICS}")
else:
    print("No new rows to append.")

# ---- Choose best per keyword and summarize ----
df = pd.read_csv(METRICS)
df = df[df["horizon_weeks"] == 6].copy()
best = (df.sort_values(["keyword","smape_canonical","rmse"])
          .groupby("keyword", as_index=False).first())

best.to_csv(BEST_OUT, index=False)

valid = best.dropna(subset=["smape_canonical","rmse"])
print(f"\nKeywords: {valid['keyword'].nunique()} / rows: {len(valid)}")
print(valid.agg(count=("rmse","count"),
                sMAPE_mean=("smape_canonical","mean"),
                sMAPE_median=("smape_canonical","median"),
                RMSE_mean=("rmse","mean"),
                RMSE_median=("rmse","median")))

for t in [10,15,20,25,30,40]:
    share = (valid["smape_canonical"] <= t).mean()
    print(f"sMAPE≤{t}: {share:.3f}")

print(f"\n✅ Saved best-per-keyword → {BEST_OUT}")


In [ ]:
# ==== Moirai H=6++ (batched, fixed-dim features, 3-context ensemble, AMP, compile) ====
# One full cell to paste into Colab.

from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ---------------- Config ----------------
FREQ           = "W"
H              = 6
NUM_SAMPLES    = 96            # per context
BATCH_SIZE     = 128
CTX_CHOICES    = [48, 64, 96]  # 3-context ensemble
TRIM_FRAC      = 0.15
NEIGH_MAX      = 3
NEIGH_LAGS     = [1,2,3,4,5,6]
FOURIER_K      = 8
RECENT_CLIP_W  = 26
BATCH_SERIES   = 64

IN_DIR  = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR = moirai_results_dir("dtw20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_H6_auto_adaptive_plus_batched.csv"

# ---------------- Device & speedups ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

try:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True
    CAN_COMPILE = True
except Exception:
    CAN_COMPILE = False

print("✅ Using device:", device)

# ---------------- IO helpers ----------------
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet", ".parq", ".pq"):
        return pl.read_parquet(p)
    if sfx == ".csv":
        return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns:
        return None
    base = (
        df_pl.select(pl.col("week"),
                     pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week").sort("ds").filter(pl.col("y").is_not_null())
    ).to_pandas()
    base["ds"] = pd.to_datetime(base["ds"])
    base["y"]  = pd.to_numeric(base["y"], errors="coerce").astype(np.float32)
    base = base.sort_values("ds").reset_index(drop=True)
    return base if len(base) >= (H + 32) else None

# ---------------- Features & metrics ----------------
def add_calendar(pdf: pd.DataFrame, k=FOURIER_K) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean()) / (idx.std() + 1e-8)
    return pdf

def get_future_cols():
    cols = []
    for i in range(1, FOURIER_K+1):
        cols += [f"cal_sin_{i}", f"cal_cos_{i}"]
    cols += ["cal_trend"]
    return cols

PAST_LAGS  = [1,2,3,4,5,6,12,26,52]  # includes YoY slot
ROLL_WINS  = [6,12,26]
NEIGH_FEAT = "neigh_agg_z"

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    m = denom != 0
    if not np.any(m):  # all zeros
        return 0.0
    return float(100.0 * np.mean(np.abs(y_pred[m] - y_true[m]) / denom[m]))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5:
        return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def neighbor_agg_signal(tr: pd.DataFrame, pdf: pd.DataFrame, k=NEIGH_MAX) -> pd.Series:
    cand = [c for c in pdf.columns if c.startswith("cpc_sim_") and "lastweek" not in c]
    if not cand:
        return pd.Series(index=pdf.index, dtype=np.float32)
    y = tr["y"]
    scored = []
    for c in cand:
        s = pd.to_numeric(tr[c], errors="coerce")
        r = y.corr(s)
        if not pd.isna(r):
            scored.append((abs(r), c))
    scored.sort(reverse=True)
    top = [c for _, c in scored[:k]]
    if not top:
        return pd.Series(index=pdf.index, dtype=np.float32)
    agg = np.zeros(len(pdf), dtype=np.float32)
    m = 0
    for c in top:
        s_full = pd.to_numeric(pdf[c], errors="coerce").astype(np.float32)
        s_tr   = s_full.iloc[:len(tr)]
        mu, sd = float(s_tr.mean()), float(s_tr.std() + 1e-8)
        z = (s_full - mu) / sd
        z = z.fillna(0.0).to_numpy(np.float32)
        agg += z; m += 1
    agg = (agg / max(m, 1)).astype(np.float32)
    return pd.Series(agg, index=pdf.index, dtype=np.float32)

# ---------------- Model ----------------
try:
    module
except NameError:
    module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()
    if CAN_COMPILE and device == "cuda":
        module = torch.compile(module)

def trimmed_median(samples: np.ndarray, trim=TRIM_FRAC):
    s = np.sort(samples, axis=0)
    n = s.shape[0]
    lo = int(np.floor(trim * n))
    hi = max(int(np.ceil((1.0 - trim) * n)), lo + 1)
    return np.median(s[lo:hi, :], axis=0).astype(np.float32)

def prep_one(pdf: pd.DataFrame):
    tr, te = split_train_test(pdf, H)
    if tr is None:
        return None
    mu, sigma = float(tr["y"].mean()), float(tr["y"].std() + 1e-8)
    pdf["y_z"] = (pdf["y"] - mu) / sigma

    for L in PAST_LAGS:
        pdf[f"y_lag_{L}"] = pdf["y_z"].shift(L)
    for W in ROLL_WINS:
        pdf[f"y_rollmean_{W}"] = pdf["y_z"].rolling(W, min_periods=1).mean().shift(1)
        pdf[f"y_rollstd_{W}"]  = pdf["y_z"].rolling(W, min_periods=2).std().shift(1)

    pdf[NEIGH_FEAT] = neighbor_agg_signal(tr, pdf, k=NEIGH_MAX)
    for L in NEIGH_LAGS:
        pdf[f"{NEIGH_FEAT}_lag{L}"] = pdf[NEIGH_FEAT].shift(L)

    warm = max(26, 52 if len(tr) >= 58 else 26)
    tr_base = pdf.iloc[:len(tr)].iloc[warm:].copy()
    te_base = pdf.iloc[len(tr):].copy()
    return tr, te, tr_base, te_base, mu, sigma

def build_entries(batch_items, ctx, past_cols, fut_cols):
    entries = []
    for tr_base, te_base in batch_items:
        entry = {
            "start": pd.Timestamp(tr_base["ds"].iloc[0]),
            "target": tr_base["y_z"].to_numpy(np.float32),
        }
        if fut_cols:
            Xp = np.stack([tr_base[c].to_numpy(np.float32) for c in fut_cols], axis=0)
            Xf = np.stack([te_base[c].to_numpy(np.float32)  for c in fut_cols], axis=0)
            entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)
        if past_cols:
            entry["past_feat_dynamic_real"] = np.stack([tr_base[c].to_numpy(np.float32) for c in past_cols], axis=0)
        entries.append(entry)
    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(fut_cols),
        past_feat_dynamic_real_dim=len(past_cols),
        context_length=ctx,
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    return predictor, ListDataset(entries, freq=FREQ)

# ---------------- Run all ----------------
print(f"Input dir: {IN_DIR}")
print(f"Output   : {OUT_CSV}")
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")

prepped, meta = [], []  # meta: (keyword, mu, sigma, y_te, y_tr)
fut_cols = get_future_cols()
past_cols = [f"y_lag_{L}" for L in PAST_LAGS] + \
            [f"y_rollmean_{W}" for W in ROLL_WINS] + \
            [f"y_rollstd_{W}"  for W in ROLL_WINS] + \
            [f"{NEIGH_FEAT}_lag{L}" for L in NEIGH_LAGS]

for p in tqdm(files, desc="prep"):
    pdf = prep_keyword_pdf(p)
    if pdf is None:
        continue
    pdf = add_calendar(pdf, k=FOURIER_K)
    try:
        res = prep_one(pdf)
        if res is None:
            continue
        tr, te, tr_base, te_base, mu, sigma = res
        prepped.append((tr_base, te_base))
        meta.append((p.stem, mu, sigma, te["y"].to_numpy(np.float32), tr["y"].to_numpy(np.float32)))
    except Exception as e:
        print(f"[WARN] {p.stem} failed in prep: {e}")

if not prepped:
    raise RuntimeError("No series prepared. Check input files and minimum length constraints.")

all_ctx_preds = [np.empty((0, H), dtype=np.float32) for _ in CTX_CHOICES]

with torch.inference_mode():
    for ctx_i, ctx in enumerate(CTX_CHOICES):
        batch_preds = []
        for i in tqdm(range(0, len(prepped), BATCH_SERIES), desc=f"ctx={ctx}"):
            chunk = prepped[i:i + BATCH_SERIES]
            predictor, ds = build_entries(chunk, ctx, past_cols, fut_cols)
            if device == "cuda":
                with torch.cuda.amp.autocast(dtype=torch.float16):
                    fc_iter = list(predictor.predict(ds))
            else:
                fc_iter = list(predictor.predict(ds))
            for fc in fc_iter:
                if fc.samples is not None:
                    zhat = trimmed_median(fc.samples, trim=TRIM_FRAC)
                else:
                    zhat = np.asarray(fc.mean, dtype=np.float32)
                batch_preds.append(zhat.astype(np.float32))
        all_ctx_preds[ctx_i] = np.vstack(batch_preds)

# average contexts and evaluate
zhat_ens = np.mean(np.stack(all_ctx_preds, axis=0), axis=0)  # [n_series, H]

rows = []
for i, (kw, mu, sigma, y_te, y_tr) in enumerate(meta):
    yhat = zhat_ens[i] * sigma + mu
    recent = y_tr[-RECENT_CLIP_W:] if len(y_tr) >= RECENT_CLIP_W else y_tr
    lo, hi = np.percentile(recent, [5, 95])
    yhat = np.clip(yhat, lo, hi).astype(np.float32)
    s = smape(y_te, yhat)
    r = rmse(y_te, yhat)
    rows.append({
        "keyword": kw,
        "exog_setting": "H6_plus_batch",
        "horizon_weeks": H,
        "smape_canonical": s,
        "rmse": r,
        "n_total": len(y_tr) + len(y_te)
    })

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print("✅ Done. Metrics:", OUT_CSV)

valid = out.dropna(subset=["smape_canonical", "rmse"])
if not valid.empty:
    print(valid.agg(count=("rmse","count"),
                    sMAPE_mean=("smape_canonical","mean"),
                    sMAPE_median=("smape_canonical","median"),
                    RMSE_mean=("rmse","mean"),
                    RMSE_median=("rmse","median")))
    for t in [10,15,20,25,30,40]:
        print(f"sMAPE≤{t:>2}: {(valid['smape_canonical'] <= t).mean():.3f}")
else:
    print("No valid rows.")


In [ ]:
# ==== Leakage-free evaluation for Moirai (H=6): train/val/test, method selection via validation only ====
# - Two methods: "fallback" (simpler) vs "advanced" (neighbor-agg + lags + calendar + log-IQR scaling).
# - Splitting: train_core (..-2H), val (H), test (H). Selection is on val only; test is held-out.
# - T4-friendly settings: no torch.compile, modest NUM_SAMPLES, short contexts.

from pathlib import Path
from datetime import date
import os, gc, numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ---------------- Config ----------------
FREQ           = "W"
H              = 6
NUM_SAMPLES    = 64            # modest sampling for T4
BATCH_SIZE     = 64
CTX_CHOICES    = [48, 64]      # short contexts to keep memory low
TRIM_FRAC      = 0.18
FOURIER_K      = 8
NEIGH_MAX      = 3
NEIGH_LAGS     = [1,2,3]       # fewer lags to cut memory

IN_DIR  = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR = moirai_results_dir("dtw20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_REPORT = OUT_DIR / "moirai_H6_leakage_free_eval.csv"

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ---------------- Device ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    torch.cuda.reset_peak_memory_stats()
print("✅ Using device:", device)

# ---------------- Helpers ----------------
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_pdf(p: Path) -> pd.DataFrame | None:
    df = _read_pl_any(p)
    if "week" not in df.columns or "cpc_week" not in df.columns:
        return None
    pdf = (
        df.select(pl.col("week"),
                  pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"),
                  *[c for c in df.columns if c not in {"week","cpc_week"}])
          .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
          .drop("week")
          .sort("ds")
          .filter(pl.col("y").is_not_null())
    ).to_pandas()
    pdf["ds"] = pd.to_datetime(pdf["ds"])
    pdf["y"]  = pd.to_numeric(pdf["y"], errors="coerce").astype(np.float32)
    pdf = pdf.sort_values("ds").reset_index(drop=True)
    return pdf if len(pdf) >= (H*3 + 32) else None  # need room for train/val/test + warmup

def add_calendar(pdf: pd.DataFrame, k=FOURIER_K) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    return pdf

def smape(y_true, y_pred):
    yt, yp = np.asarray(y_true, np.float32), np.asarray(y_pred, np.float32)
    denom = (np.abs(yt)+np.abs(yp))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(yp[m]-yt[m])/denom[m])) if np.any(m) else 0.0

def rmse(y_true, y_pred):
    yt, yp = np.asarray(y_true, np.float32), np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((yp-yt)**2)))

def detect_neighbor_cols(pdf: pd.DataFrame) -> list[str]:
    cand = [c for c in pdf.columns if "cpc" in c.lower() and c.lower() != "cpc_week"]
    keep = []
    for c in cand:
        s = pd.to_numeric(pdf[c], errors="coerce")
        if s.notna().sum() >= 16:
            pdf[c] = s.astype(np.float32)
            keep.append(c)
    return keep[:20]

def neighbor_agg_signal(tr: pd.DataFrame, pdf: pd.DataFrame, neigh_cols: list[str], k=NEIGH_MAX) -> pd.Series:
    if not neigh_cols:
        return pd.Series(index=pdf.index, dtype=np.float32)
    y = tr["y"]
    scored = []
    for c in neigh_cols:
        r = y.corr(pd.to_numeric(tr[c], errors="coerce"))
        if not pd.isna(r): scored.append((abs(r), c))
    scored.sort(reverse=True)
    top = [c for _,c in scored[:k]]
    if not top:
        return pd.Series(index=pdf.index, dtype=np.float32)
    agg = np.zeros(len(pdf), dtype=np.float32); m = 0
    for c in top:
        s_full = pd.to_numeric(pdf[c], errors="coerce").astype(np.float32)
        s_tr   = s_full.iloc[:len(tr)]
        mu, sd = float(s_tr.mean()), float(s_tr.std() + 1e-8)
        z = (s_full - mu) / sd
        z = np.nan_to_num(z, nan=0.0).astype(np.float32)
        agg += z; m += 1
    return pd.Series((agg/max(m,1)).astype(np.float32), index=pdf.index)

def trimmed_median(samples: np.ndarray, trim=TRIM_FRAC):
    s = np.sort(samples, axis=0)
    n = s.shape[0]
    lo = int(np.floor(trim * n))
    hi = max(int(np.ceil((1.0 - trim) * n)), lo + 1)
    return np.median(s[lo:hi, :], axis=0).astype(np.float32)

# ---------------- Model ----------------
try:
    module
except NameError:
    module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def moirai_predict(tr_base, te_base, past_cols, fut_cols, ctx, num_samples=NUM_SAMPLES):
    entry = {"start": pd.Timestamp(tr_base["ds"].iloc[0]),
             "target": tr_base["y_z"].to_numpy(np.float32)}
    if fut_cols:
        Xp = np.stack([tr_base[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        Xf = np.stack([te_base[c].to_numpy(np.float32)  for c in fut_cols], axis=0)
        entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)
    if past_cols:
        entry["past_feat_dynamic_real"] = np.stack([tr_base[c].to_numpy(np.float32) for c in past_cols], axis=0)

    forecaster = MoiraiForecast(
        prediction_length=len(te_base),
        target_dim=1,
        feat_dynamic_real_dim=len(fut_cols),
        past_feat_dynamic_real_dim=len(past_cols),
        context_length=ctx,
        module=module,
        patch_size="auto",
        num_samples=num_samples,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    ds = ListDataset([entry], freq=FREQ)
    if device == "cuda":
        with torch.amp.autocast('cuda', dtype=torch.float16):
            fc = next(predictor.predict(ds))
    else:
        fc = next(predictor.predict(ds))
    if fc.samples is not None:
        zhat = trimmed_median(fc.samples)
    else:
        zhat = np.asarray(fc.mean, dtype=np.float32)
    return zhat

# ---------------- Two methods (no test leakage) ----------------
def build_frames(pdf, n_train):
    """Return train frame (0..n_train-1) and horizon frame (n_train..n_train+H-1)."""
    tr = pdf.iloc[:n_train].copy()
    te = pdf.iloc[n_train:n_train+H].copy()
    return tr, te

def run_method(pdf: pd.DataFrame, n_train: int, method: str):
    """
    Build features using ONLY data up to n_train (no peeking), forecast next H.
    Returns yhat (original space) for the horizon [n_train : n_train+H).
    """
    assert method in ("fallback","advanced")
    # Add calendar (future-known)
    pdf2 = add_calendar(pdf)

    # Train slice
    tr, te = build_frames(pdf2, n_train)
    if len(te) < H or len(tr) < 16:
        return None

    # Scaling on TRAIN ONLY
    if method == "advanced":
        # log-IQR scaling
        eps = 1e-6
        tr_log = np.log1p(np.maximum(tr["y"].values, 0) + eps).astype(np.float32)
        med = float(np.median(tr_log))
        iqr = float(np.percentile(tr_log, 75) - np.percentile(tr_log, 25) + 1e-6)
        def z(y): return (np.log1p(np.maximum(y, 0)+eps).astype(np.float32) - med)/iqr
        pdf2["y_z"] = z(pdf2["y"].values)
    else:
        # plain z-norm
        mu, sigma = float(tr["y"].mean()), float(tr["y"].std() + 1e-8)
        pdf2["y_z"] = (pdf2["y"] - mu) / sigma

    # Build target-based past features on TRAIN (shift ensures no future leakage)
    # Moderate set for T4
    for L in [1,2,3,4,5,6,12,26,52]:
        pdf2[f"y_lag_{L}"] = pdf2["y_z"].shift(L)
    for W in [6,12,26]:
        pdf2[f"y_rollmean_{W}"] = pdf2["y_z"].rolling(W, min_periods=1).mean().shift(1)
        pdf2[f"y_rollstd_{W}"]  = pdf2["y_z"].rolling(W, min_periods=2).std().shift(1)

    # Neighbor aggregated z-signal (rank on TRAIN only), then lags (past-only)
    neigh_cols = detect_neighbor_cols(pdf2)
    nb = neighbor_agg_signal(tr, pdf2, neigh_cols, k=NEIGH_MAX)
    pdf2["neigh_agg_z"] = nb
    for L in NEIGH_LAGS:
        pdf2[f"neigh_agg_z_lag{L}"] = pdf2["neigh_agg_z"].shift(L)

    # Warmup to make lags valid, then cut TRAIN and assemble
    warm = max(26, 52 if len(tr) >= 58 else 26)
    tr_base = pdf2.iloc[:n_train].iloc[warm:].copy()
    te_base = pdf2.iloc[n_train:n_train+H].copy()

    past_cols = [c for c in tr_base.columns if c.startswith(("y_lag_","y_rollmean_","y_rollstd_","neigh_agg_z_lag"))]
    fut_cols  = [c for c in pdf2.columns if c.startswith(("cal_sin_","cal_cos_","cal_trend"))]

    # Short-context ensemble without rebuilding scalers
    preds = []
    for ctx in CTX_CHOICES:
        try:
            zhat = moirai_predict(tr_base, te_base, past_cols, fut_cols, ctx, num_samples=NUM_SAMPLES)
            preds.append(zhat.astype(np.float32))
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache(); gc.collect()
                continue
            else:
                raise
    if not preds:
        return None
    z_mean = np.mean(np.stack(preds, axis=0), axis=0)

    # Inverse scaling to original space
    if method == "advanced":
        yhat = np.expm1(z_mean*iqr + med) - 1e-6
        yhat = np.maximum(yhat, 0).astype(np.float32)
    else:
        yhat = z_mean * (sigma) + mu

    # light clipping on recent train to stabilize sMAPE
    recent = tr["y"].to_numpy(np.float32)[-26:] if len(tr) >= 26 else tr["y"].to_numpy(np.float32)
    lo, hi = np.percentile(recent, [5,95]) if len(recent) else (None, None)
    if lo is not None:
        yhat = np.clip(yhat, lo, hi)

    return yhat.astype(np.float32)

# ---------------- Evaluation (leakage-free) ----------------
module = module.eval()  # just to be explicit

files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} series.")

rows = []
for p in tqdm(files, desc="eval"):
    pdf = prep_pdf(p)
    if pdf is None:
        continue

    N = len(pdf)
    n_train_core = N - 2*H
    n_train_full = N - H
    if n_train_core <= 32:
        continue  # not enough history

    # 1) VALIDATION forecasts from TRAIN_CORE only
    yhat_fb_val = run_method(pdf, n_train_core, method="fallback")
    yhat_adv_val= run_method(pdf, n_train_core, method="advanced")
    y_val       = pdf["y"].iloc[n_train_core:n_train_core+H].to_numpy(np.float32)

    if yhat_fb_val is None or yhat_adv_val is None:
        continue

    s_fb_val = smape(y_val, yhat_fb_val); r_fb_val = rmse(y_val, yhat_fb_val)
    s_ad_val = smape(y_val, yhat_adv_val); r_ad_val = rmse(y_val, yhat_adv_val)

    # 2) TEST forecasts from TRAIN_CORE+VAL (NO peeking at test)
    yhat_fb_test = run_method(pdf, n_train_full, method="fallback")
    yhat_adv_test= run_method(pdf, n_train_full, method="advanced")
    y_test       = pdf["y"].iloc[n_train_full:n_train_full+H].to_numpy(np.float32)

    if yhat_fb_test is None or yhat_adv_test is None:
        continue

    # Record everything so we can compute both strategies
    s_fb_test = smape(y_test, yhat_fb_test); r_fb_test = rmse(y_test, yhat_fb_test)
    s_ad_test = smape(y_test, yhat_adv_test); r_ad_test = rmse(y_test, yhat_adv_test)

    rows.append({
        "keyword": p.stem,
        "sMAPE_val_fallback": s_fb_val, "RMSE_val_fallback": r_fb_val,
        "sMAPE_val_advanced": s_ad_val, "RMSE_val_advanced": r_ad_val,
        "sMAPE_test_fallback": s_fb_test, "RMSE_test_fallback": r_fb_test,
        "sMAPE_test_advanced": s_ad_test, "RMSE_test_advanced": r_ad_test,
    })

res = pd.DataFrame(rows)
res.to_csv(OUT_REPORT, index=False)
print(f"✅ Wrote per-series validation/test metrics → {OUT_REPORT}")

# ---------------- Summaries ----------------
def summarize_series(mask, label):
    sub = res.loc[mask].copy()
    if sub.empty:
        print(f"{label}: 0 series")
        return
    def shares(x):
        return {f"sMAPE≤{t}": float((x <= t).mean()) for t in [10,15,20,25,30,40]}
    print(f"\n=== {label} ===")
    print(f"Series: {len(sub)}")
    print("fallback test:",
          {"mean": float(sub['sMAPE_test_fallback'].mean()),
           "median": float(sub['sMAPE_test_fallback'].median()),
           **shares(sub['sMAPE_test_fallback'])})
    print("advanced test:",
          {"mean": float(sub['sMAPE_test_advanced'].mean()),
           "median": float(sub['sMAPE_test_advanced'].median()),
           **shares(sub['sMAPE_test_advanced'])})

# Strategy A (GLOBAL): pick a single method by avg validation sMAPE across all series
avg_val_fb = float(res["sMAPE_val_fallback"].mean())
avg_val_ad = float(res["sMAPE_val_advanced"].mean())
global_pick = "advanced" if avg_val_ad < avg_val_fb else "fallback"
print(f"\n[GLOBAL] Average validation sMAPE — fallback: {avg_val_fb:.3f}, advanced: {avg_val_ad:.3f} → pick: {global_pick}")

if global_pick == "advanced":
    global_test = res["sMAPE_test_advanced"]
else:
    global_test = res["sMAPE_test_fallback"]

print("[GLOBAL] Test sMAPE:",
      {"mean": float(global_test.mean()),
       "median": float(global_test.median()),
       "sMAPE≤10": float((global_test<=10).mean()),
       "sMAPE≤15": float((global_test<=15).mean()),
       "sMAPE≤20": float((global_test<=20).mean()),
       "sMAPE≤25": float((global_test<=25).mean()),
       "sMAPE≤30": float((global_test<=30).mean()),
       "sMAPE≤40": float((global_test<=40).mean())})

# Strategy B (PER-SERIES): pick per keyword by validation sMAPE (still leakage-free)
per_series_pick_adv = res["sMAPE_val_advanced"] < res["sMAPE_val_fallback"]
test_ps = np.where(per_series_pick_adv, res["sMAPE_test_advanced"], res["sMAPE_test_fallback"])
test_ps = pd.Series(test_ps)

print("\n[PER-SERIES] Validation picked advanced on",
      f"{int(per_series_pick_adv.sum())}/{len(res)} series ({per_series_pick_adv.mean():.1%})")

print("[PER-SERIES] Test sMAPE:",
      {"mean": float(test_ps.mean()),
       "median": float(test_ps.median()),
       "sMAPE≤10": float((test_ps<=10).mean()),
       "sMAPE≤15": float((test_ps<=15).mean()),
       "sMAPE≤20": float((test_ps<=20).mean()),
       "sMAPE≤25": float((test_ps<=25).mean()),
       "sMAPE≤30": float((test_ps<=30).mean()),
       "sMAPE≤40": float((test_ps<=40).mean())})

# Optional: print separate summaries for convenience
summarize_series(mask=np.full(len(res), True), label="All series (method-wise test comparison)")


### ↓ Canonical cell for the paper (see banner at the top of the notebook)

Below is the cell whose output is reported in the paper. All earlier cells in this notebook run, pin packages, or iterate the methodology that leads up to this cell.


In [ ]:
# ==== FIXED: Leakage-free meta-ensemble (no-NaN), validation-derived weights, honest test ====
# Works fast on A100; T4: set CTX_CHOICES=[48,64], NUM_SAMPLES=48, BATCH_SERIES=48.

from pathlib import Path
from datetime import date
import os, gc, numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ---------------- Config ----------------
FREQ           = "W"
H              = 6
NUM_SAMPLES    = 96
BATCH_SIZE     = 128
CTX_CHOICES    = [48, 64, 96]     # T4: [48,64]
TRIM_FRAC      = 0.15
FOURIER_K      = 8
NEIGH_MAX      = 3
RECENT_CLIP_W  = 26               # used only for final blend safety if needed
BATCH_SERIES   = 96               # T4: 48
WEIGHT_P       = 2.0              # weights ~ (err+eps)^(-p)
EPS_ERR        = 1e-6

IN_DIR   = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR  = moirai_results_dir("dtw20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_METRICS = OUT_DIR / "moirai_H6_meta_ensemble_metrics.csv"
OUT_WEIGHTS = OUT_DIR / "moirai_H6_meta_ensemble_weights.csv"

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ---------------- Device ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    torch.cuda.reset_peak_memory_stats()
print("✅ Using device:", device)

# ---------------- Helpers ----------------
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-"); return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError("Unsupported file type")

def prep_pdf(p: Path) -> pd.DataFrame | None:
    df = _read_pl_any(p)
    if "week" not in df.columns or "cpc_week" not in df.columns: return None
    pdf = (df.select(pl.col("week"),
                     pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"),
                     *[c for c in df.columns if c not in {"week","cpc_week"}])
             .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
             .drop("week").sort("ds").filter(pl.col("y").is_not_null())
           ).to_pandas()
    pdf["ds"] = pd.to_datetime(pdf["ds"])
    pdf["y"]  = pd.to_numeric(pdf["y"], errors="coerce").astype(np.float32)
    pdf = pdf.sort_values("ds").reset_index(drop=True)
    # need train_core + val + test + warmup
    return pdf if len(pdf) >= (H*3 + 40) else None

def add_calendar(pdf: pd.DataFrame, k=FOURIER_K) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int); period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    return pdf

def smape(y_true, y_pred):
    yt, yp = np.asarray(y_true, np.float32), np.asarray(y_pred, np.float32)
    denom = (np.abs(yt)+np.abs(yp))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(yp[m]-yt[m])/denom[m])) if np.any(m) else 0.0

def rmse(y_true, y_pred):
    yt, yp = np.asarray(y_true, np.float32), np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((yp-yt)**2)))

def detect_neighbor_cols(pdf: pd.DataFrame) -> list[str]:
    cand = [c for c in pdf.columns if "cpc" in c.lower() and c.lower() != "cpc_week"]
    keep = []
    for c in cand:
        s = pd.to_numeric(pdf[c], errors="coerce")
        if s.notna().sum() >= 16:
            pdf[c] = s.astype(np.float32)
            keep.append(c)
    return keep[:20]

def neighbor_agg_signal(tr: pd.DataFrame, pdf: pd.DataFrame, neigh_cols: list[str], k=NEIGH_MAX) -> pd.Series:
    if not neigh_cols: return pd.Series(index=pdf.index, dtype=np.float32)
    y = tr["y"]; scored = []
    for c in neigh_cols:
        r = y.corr(pd.to_numeric(tr[c], errors="coerce"))
        if not pd.isna(r): scored.append((abs(r), c))
    scored.sort(reverse=True); top = [c for _,c in scored[:k]]
    if not top: return pd.Series(index=pdf.index, dtype=np.float32)
    agg = np.zeros(len(pdf), dtype=np.float32); m = 0
    for c in top:
        s_full = pd.to_numeric(pdf[c], errors="coerce").astype(np.float32)
        s_tr   = s_full.iloc[:len(tr)]
        mu, sd = float(s_tr.mean()), float(s_tr.std() + 1e-8)
        z = (s_full - mu) / sd
        z = np.nan_to_num(z, nan=0.0).astype(np.float32)
        agg += z; m += 1
    return pd.Series((agg/max(m,1)).astype(np.float32), index=pdf.index)

def trimmed_median(samples: np.ndarray, trim=TRIM_FRAC):
    s = np.sort(samples, axis=0); n = s.shape[0]
    lo = int(np.floor(trim * n)); hi = max(int(np.ceil((1.0 - trim) * n)), lo + 1)
    return np.median(s[lo:hi, :], axis=0).astype(np.float32)

# ---------------- Model ----------------
try:
    module
except NameError:
    module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def moirai_predict_batch(batch_items, ctx, num_samples=NUM_SAMPLES):
    """Return list of z-space predictions for each (tr_base, te_base, past_cols, fut_cols)."""
    entries = []
    for tr_base, te_base, past_cols, fut_cols in batch_items:
        entry = {"start": pd.Timestamp(tr_base["ds"].iloc[0]),
                 "target": tr_base["y_z"].to_numpy(np.float32)}
        if fut_cols:
            Xp = np.stack([tr_base[c].to_numpy(np.float32) for c in fut_cols], axis=0)
            Xf = np.stack([te_base[c].to_numpy(np.float32)  for c in fut_cols], axis=0)
            entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)
        if past_cols:
            entry["past_feat_dynamic_real"] = np.stack([tr_base[c].to_numpy(np.float32) for c in past_cols], axis=0)
        entries.append(entry)

    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(batch_items[0][3]),
        past_feat_dynamic_real_dim=len(batch_items[0][2]),
        context_length=ctx,
        module=module,
        patch_size="auto",
        num_samples=num_samples,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    ds = ListDataset(entries, freq=FREQ)

    preds = []
    if device == "cuda":
        with torch.amp.autocast('cuda', dtype=torch.float16):
            for fc in predictor.predict(ds):
                preds.append(trimmed_median(fc.samples) if fc.samples is not None
                             else np.asarray(fc.mean, dtype=np.float32))
    else:
        for fc in predictor.predict(ds):
            preds.append(trimmed_median(fc.samples) if fc.samples is not None
                         else np.asarray(fc.mean, dtype=np.float32))
    return preds

# ---------------- Build features for two horizons (val & test) ----------------
PAST_LAGS  = [1,2,3,4,5,6,12,26,52]
ROLL_WINS  = [6,12,26]

def build_feature_frames(pdf, n_train):
    """Use only 0..n_train-1 to fit scalers/features; forecast next H."""
    pdf2 = add_calendar(pdf)
    tr   = pdf2.iloc[:n_train].copy()
    te   = pdf2.iloc[n_train:n_train+H].copy()
    if len(te) < H or len(tr) < 16: return None

    # robust scaling (log-IQR) on TRAIN only for Moirai target
    eps = 1e-6
    tr_log = np.log1p(np.maximum(tr["y"].values, 0) + eps).astype(np.float32)
    med = float(np.median(tr_log))
    iqr = float(np.percentile(tr_log, 75) - np.percentile(tr_log, 25) + 1e-6)
    pdf2["y_z"] = (np.log1p(np.maximum(pdf2["y"].values, 0) + eps).astype(np.float32) - med)/iqr

    # lags & rolls (past-only)
    for L in PAST_LAGS:
        pdf2[f"y_lag_{L}"] = pdf2["y_z"].shift(L)
    for W in ROLL_WINS:
        pdf2[f"y_rollmean_{W}"] = pdf2["y_z"].rolling(W, min_periods=1).mean().shift(1)
        pdf2[f"y_rollstd_{W}"]  = pdf2["y_z"].rolling(W, min_periods=2).std().shift(1)

    # neighbor agg (OPTIONAL; comment out if you want purely univariate)
    # neigh_cols = detect_neighbor_cols(pdf2)
    # nb = neighbor_agg_signal(tr, pdf2, neigh_cols, k=NEIGH_MAX)
    # pdf2["neigh_agg_z"] = nb
    # for L in [1,2,3,4,5,6]:
    #     pdf2[f"neigh_agg_z_lag{L}"] = pdf2["neigh_agg_z"].shift(L)

    warm = max(26, 52 if len(tr) >= 58 else 26)
    tr_base = pdf2.iloc[:n_train].iloc[warm:].copy()
    te_base = pdf2.iloc[n_train:n_train+H].copy()

    past_cols = [c for c in tr_base.columns if c.startswith(("y_lag_","y_rollmean_","y_rollstd_"))]
    fut_cols  = [c for c in pdf2.columns if c.startswith(("cal_sin_","cal_cos_","cal_trend"))]
    scalers = {"med": med, "iqr": iqr}
    return tr, te, tr_base, te_base, past_cols, fut_cols, scalers

# ---------------- Baselines (no leakage) ----------------
def seasonal_naive(y_hist, period=52):
    if len(y_hist) < period:  # fallback to last
        return np.repeat(y_hist[-1], H).astype(np.float32)
    return y_hist[-period:][:H].astype(np.float32)

def last_value(y_hist):
    return np.repeat(y_hist[-1], H).astype(np.float32)

def drift_naive(y_hist, k=13):
    k = min(k, len(y_hist))
    x = np.arange(k, dtype=np.float32)
    y = y_hist[-k:].astype(np.float32)
    x_mean = x.mean(); y_mean = y.mean()
    denom = np.sum((x - x_mean)**2) + 1e-8
    slope = float(np.sum((x - x_mean)*(y - y_mean)) / denom)
    intercept = float(y_mean - slope*x_mean)
    steps = np.arange(1, H+1, dtype=np.float32)
    return (intercept + slope*(x[-1] + steps)).astype(np.float32)

# ---------------- Main pipeline ----------------
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")

meta = []  # per series bookkeeping
val_batches, test_batches = [], []  # items for Moirai val/test prediction
y_val_list, y_test_list = [], []
naive_val, naive_test = [], []

for p in tqdm(files, desc="prep"):
    pdf = prep_pdf(p)
    if pdf is None: continue
    N = len(pdf)
    n_train_core = N - 2*H
    n_train_full = N - H
    if n_train_core <= 40: continue

    # VAL setup (train_core -> val)
    res_v = build_feature_frames(pdf, n_train_core)
    # TEST setup (train_core+val -> test)
    res_t = build_feature_frames(pdf, n_train_full)
    if res_v is None or res_t is None: continue
    tr_v, te_v, trb_v, teb_v, past_v, fut_v, sc_v = res_v
    tr_t, te_t, trb_t, teb_t, past_t, fut_t, sc_t = res_t

    # record for batched Moirai
    val_batches.append((trb_v, teb_v, past_v, fut_v))
    test_batches.append((trb_t, teb_t, past_t, fut_t))

    # ground truths
    y_val_list.append(te_v["y"].to_numpy(np.float32))
    y_test_list.append(te_t["y"].to_numpy(np.float32))

    # simple baselines (orig space) for val/test
    y_hist_v = tr_v["y"].to_numpy(np.float32)
    y_hist_t = tr_t["y"].to_numpy(np.float32)
    naive_val.append({"seasonal": seasonal_naive(y_hist_v),
                      "last": last_value(y_hist_v),
                      "drift": drift_naive(y_hist_v, k=13)})
    naive_test.append({"seasonal": seasonal_naive(y_hist_t),
                       "last": last_value(y_hist_t),
                       "drift": drift_naive(y_hist_t, k=13)})

    meta.append({"keyword": p.stem, "sc_val": sc_v, "sc_test": sc_t})

# Run Moirai in batches for each context for VAL and TEST, then ensemble (simple equal-weight across contexts)
def run_all_batches(batches, label):
    preds_ctx = []
    for ctx in CTX_CHOICES:
        ctx_preds = []
        for i in tqdm(range(0, len(batches), BATCH_SERIES), desc=f"{label} ctx={ctx}"):
            chunk = batches[i:i+BATCH_SERIES]
            out = moirai_predict_batch(chunk, ctx, num_samples=NUM_SAMPLES)
            ctx_preds.extend(out)
        preds_ctx.append(np.stack(ctx_preds, axis=0))
    return np.mean(np.stack(preds_ctx, axis=0), axis=0)  # [n_series, H]

print("Predicting Moirai on validation...")
zhat_val = run_all_batches(val_batches, "val")
print("Predicting Moirai on test...")
zhat_test = run_all_batches(test_batches, "test")

# invert scaling back to original space (per series)
yhat_m_val  = []
yhat_m_test = []
for i, m in enumerate(meta):
    med_v, iqr_v = m["sc_val"]["med"], m["sc_val"]["iqr"]
    med_t, iqr_t = m["sc_test"]["med"], m["sc_test"]["iqr"]
    yhat_m_val.append(np.expm1(zhat_val[i]*iqr_v + med_v) - 1e-6)
    yhat_m_test.append(np.expm1(zhat_test[i]*iqr_t + med_t) - 1e-6)
yhat_m_val  = np.maximum(np.vstack(yhat_m_val), 0.0).astype(np.float32)
yhat_m_test = np.maximum(np.vstack(yhat_m_test), 0.0).astype(np.float32)

# ---------------- Validation errors & weights ----------------
def smape_vec(y, yhat):
    denom = (np.abs(y)+np.abs(yhat))/2.0
    m = denom != 0
    out = np.zeros(y.shape[0], dtype=np.float32)
    for i in range(y.shape[0]):
        mi = m[i]
        out[i] = float(100.0*np.mean(np.abs(yhat[i,mi]-y[i,mi])/denom[i,mi])) if np.any(mi) else 0.0
    return out

yval = np.stack(y_val_list, axis=0)   # [n, H]
ytst = np.stack(y_test_list, axis=0)  # [n, H]

# components: Moirai (equal-weighted contexts), seasonal, last, drift
C = 4
def stack_components(which="val"):
    arr = []
    for i in range(len(meta)):
        if which == "val":
            comps = [yhat_m_val[i], naive_val[i]["seasonal"], naive_val[i]["last"], naive_val[i]["drift"]]
        else:
            comps = [yhat_m_test[i], naive_test[i]["seasonal"], naive_test[i]["last"], naive_test[i]["drift"]]
        arr.append(np.stack(comps, axis=0))
    return np.stack(arr, axis=0)  # [n, C, H]

Xval = stack_components("val")
Xtst = stack_components("test")

val_errs = np.zeros((len(meta), C), dtype=np.float32)
for j in range(C):
    val_errs[:,j] = smape_vec(yval, Xval[:,j,:])

# weights from validation: w_j ~ (err_j + eps)^(-p)
W = (np.maximum(val_errs, 0.0) + EPS_ERR) ** (-WEIGHT_P)
W = W / W.sum(axis=1, keepdims=True)

# ---------------- Build final blended TEST forecast & evaluate ----------------
Yblend = (Xtst * W[:,:,None]).sum(axis=1)  # [n, H]

# conservative clipping using baseline envelope (keeps magnitudes sane, no leakage)
for i in range(len(meta)):
    env = np.concatenate([naive_test[i]["seasonal"], naive_test[i]["last"], naive_test[i]["drift"]]).astype(np.float32)
    lo, hi = np.percentile(env, [5,95])
    Yblend[i,:] = np.clip(Yblend[i,:], lo, hi)

# Evaluate
s_meta = smape_vec(ytst, Yblend)
r_meta = np.sqrt(((ytst - Yblend)**2).mean(axis=1)).astype(np.float32)
s_moir = smape_vec(ytst, yhat_m_test)

def shares(x):
    return {f"sMAPE≤{t}": float((x <= t).mean()) for t in [10,15,20,25,30,40]}

print("\n=== Leakage-free meta-ensemble (TEST) — fixed ===")
print({"mean": float(s_meta.mean()),
       "median": float(np.median(s_meta)),
       **shares(s_meta)})

print("\nMoirai-only (same TEST window):")
print({"mean": float(s_moir.mean()),
       "median": float(np.median(s_moir)),
       **shares(s_moir)})

# ---------------- Save per-series outputs ----------------
weights_df = pd.DataFrame(W, columns=["w_moirai","w_seasonal","w_last","w_drift"])
weights_df.insert(0, "keyword", [m["keyword"] for m in meta])

metrics_df = pd.DataFrame({
    "keyword": [m["keyword"] for m in meta],
    "sMAPE_test_meta": s_meta,
    "RMSE_test_meta":  r_meta,
    "sMAPE_test_moirai": s_moir,
})
for t in [10,15,20,25,30,40]:
    metrics_df[f"hit_meta_le_{t}"] = (s_meta <= t).astype(int)

weights_df.to_csv(OUT_WEIGHTS, index=False)
metrics_df.to_csv(OUT_METRICS, index=False)
print(f"\n✅ Saved per-series weights → {OUT_WEIGHTS}")
print(f"✅ Saved per-series test metrics → {OUT_METRICS}")

if device == "cuda":
    try:
        peak = torch.cuda.max_memory_allocated() / (1024**3)
        print(f"[GPU] Peak VRAM: {peak:.2f} GiB")
    except Exception:
        pass


In [ ]:
# ==== Leakage-free per-series ctx selection + gated blend (protects Moirai from weak baselines) ====
# A100-ready. On T4: set CTX_CHOICES = [48, 64], NUM_SAMPLES=48, BATCH_SERIES=48.

from pathlib import Path
from datetime import date
import os, numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch, gc
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ---------------- Config ----------------
FREQ           = "W"
H              = 6
NUM_SAMPLES    = 96
BATCH_SIZE     = 128
BATCH_SERIES   = 96         # T4: 48
CTX_CHOICES    = [48, 64, 96]  # T4: [48, 64]
TRIM_FRAC      = 0.15
FOURIER_K      = 8

# Blend hyperparams (conservative)
WEIGHT_P       = 2.0        # w ~ (err+eps)^(-p)
EPS_ERR        = 1e-6
MOIRAI_W_MIN   = 0.50       # never let Moirai be < 50% in the blend
DELTA_MIN      = 0.20       # require val-blend sMAPE improvement vs Moirai (in absolute points) to accept blend
USE_CLIP       = False      # if True: envelope clipping to [1,99] pct of baselines on TEST

IN_DIR   = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR  = moirai_results_dir("dtw20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_METRICS = OUT_DIR / "moirai_H6_ctxselect_gatedblend_metrics.csv"
OUT_WEIGHTS = OUT_DIR / "moirai_H6_ctxselect_gatedblend_weights.csv"

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ---------------- Device ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    torch.cuda.reset_peak_memory_stats()
print("✅ Using device:", device)

# ---------------- Helpers ----------------
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-"); return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError("Unsupported file type")

def prep_pdf(p: Path) -> pd.DataFrame | None:
    df = _read_pl_any(p)
    if "week" not in df.columns or "cpc_week" not in df.columns: return None
    pdf = (df.select(pl.col("week"),
                     pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
             .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
             .drop("week").sort("ds").filter(pl.col("y").is_not_null())
           ).to_pandas()
    pdf["ds"] = pd.to_datetime(pdf["ds"])
    pdf["y"]  = pd.to_numeric(pdf["y"], errors="coerce").astype(np.float32)
    pdf = pdf.sort_values("ds").reset_index(drop=True)
    # need train_core + val + test + warmup
    return pdf if len(pdf) >= (H*3 + 40) else None

def add_calendar(pdf: pd.DataFrame, k=FOURIER_K) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int); period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    return pdf

def smape(y_true, y_pred):
    yt, yp = np.asarray(y_true, np.float32), np.asarray(y_pred, np.float32)
    denom = (np.abs(yt)+np.abs(yp))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(yp[m]-yt[m])/denom[m])) if np.any(m) else 0.0

def rmse(y_true, y_pred):
    yt, yp = np.asarray(y_true, np.float32), np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((yp-yt)**2)))

def trimmed_median(samples: np.ndarray, trim=TRIM_FRAC):
    s = np.sort(samples, axis=0); n = s.shape[0]
    lo = int(np.floor(trim*n)); hi = max(int(np.ceil((1.0-trim)*n)), lo+1)
    return np.median(s[lo:hi, :], axis=0).astype(np.float32)

# ---------------- Model ----------------
try:
    module
except NameError:
    module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def moirai_predict_batch(batch_items, ctx, num_samples=NUM_SAMPLES):
    """Return list of z-space predictions for each (tr_base, te_base, past_cols, fut_cols)."""
    entries = []
    for tr_base, te_base, past_cols, fut_cols in batch_items:
        entry = {"start": pd.Timestamp(tr_base["ds"].iloc[0]),
                 "target": tr_base["y_z"].to_numpy(np.float32)}
        if fut_cols:
            Xp = np.stack([tr_base[c].to_numpy(np.float32) for c in fut_cols], axis=0)
            Xf = np.stack([te_base[c].to_numpy(np.float32)  for c in fut_cols], axis=0)
            entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)
        if past_cols:
            entry["past_feat_dynamic_real"] = np.stack([tr_base[c].to_numpy(np.float32) for c in past_cols], axis=0)
        entries.append(entry)

    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(batch_items[0][3]),
        past_feat_dynamic_real_dim=len(batch_items[0][2]),
        context_length=ctx,
        module=module,
        patch_size="auto",
        num_samples=num_samples,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    ds = ListDataset(entries, freq=FREQ)

    preds = []
    if device == "cuda":
        with torch.amp.autocast('cuda', dtype=torch.float16):
            for fc in predictor.predict(ds):
                preds.append(trimmed_median(fc.samples) if fc.samples is not None
                             else np.asarray(fc.mean, dtype=np.float32))
    else:
        for fc in predictor.predict(ds):
            preds.append(trimmed_median(fc.samples) if fc.samples is not None
                         else np.asarray(fc.mean, dtype=np.float32))
    return preds

# ---------------- Features (log-IQR target, past-only lags) ----------------
PAST_LAGS  = [1,2,3,4,5,6,12,26,52]
ROLL_WINS  = [6,12,26]

def build_feature_frames(pdf, n_train):
    """Use only 0..n_train-1 to fit scalers/features; forecast next H."""
    pdf2 = add_calendar(pdf)
    tr   = pdf2.iloc[:n_train].copy()
    te   = pdf2.iloc[n_train:n_train+H].copy()
    if len(te) < H or len(tr) < 16: return None

    # robust scaling (log-IQR) on TRAIN only
    eps = 1e-6
    tr_log = np.log1p(np.maximum(tr["y"].values, 0) + eps).astype(np.float32)
    med = float(np.median(tr_log))
    iqr = float(np.percentile(tr_log, 75) - np.percentile(tr_log, 25) + 1e-6)
    pdf2["y_z"] = (np.log1p(np.maximum(pdf2["y"].values, 0) + eps).astype(np.float32) - med)/iqr

    # past-only features
    for L in PAST_LAGS:
        pdf2[f"y_lag_{L}"] = pdf2["y_z"].shift(L)
    for W in ROLL_WINS:
        pdf2[f"y_rollmean_{W}"] = pdf2["y_z"].rolling(W, min_periods=1).mean().shift(1)
        pdf2[f"y_rollstd_{W}"]  = pdf2["y_z"].rolling(W, min_periods=2).std().shift(1)

    warm = max(26, 52 if len(tr) >= 58 else 26)
    tr_base = pdf2.iloc[:n_train].iloc[warm:].copy()
    te_base = pdf2.iloc[n_train:n_train+H].copy()

    past_cols = [c for c in tr_base.columns if c.startswith(("y_lag_","y_rollmean_","y_rollstd_"))]
    fut_cols  = [c for c in pdf2.columns if c.startswith(("cal_sin_","cal_cos_","cal_trend"))]
    scalers   = {"med": med, "iqr": iqr}
    return tr, te, tr_base, te_base, past_cols, fut_cols, scalers

# ---------------- Baselines ----------------
def seasonal_naive(y_hist, period=52):
    if len(y_hist) < period: return np.repeat(y_hist[-1], H).astype(np.float32)
    return y_hist[-period:][:H].astype(np.float32)

def last_value(y_hist):
    return np.repeat(y_hist[-1], H).astype(np.float32)

def drift_naive(y_hist, k=13):
    k = min(k, len(y_hist))
    x = np.arange(k, dtype=np.float32); y = y_hist[-k:].astype(np.float32)
    xm, ym = x.mean(), y.mean()
    slope = float(np.sum((x-xm)*(y-ym)) / (np.sum((x-xm)**2)+1e-8))
    intercept = float(ym - slope*xm)
    steps = np.arange(1, H+1, dtype=np.float32)
    return (intercept + slope*(x[-1] + steps)).astype(np.float32)

def smape_vec(y, yhat):
    denom = (np.abs(y)+np.abs(yhat))/2.0
    m = denom != 0
    out = np.zeros(y.shape[0], dtype=np.float32)
    for i in range(y.shape[0]):
        mi = m[i]
        out[i] = float(100.0*np.mean(np.abs(yhat[i,mi]-y[i,mi])/denom[i,mi])) if np.any(mi) else 0.0
    return out

# ---------------- Prep series ----------------
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")

meta = []
y_val, y_tst = [], []
val_items_grid = {ctx: [] for ctx in CTX_CHOICES}
tst_items_grid = {ctx: [] for ctx in CTX_CHOICES}
nb_val, nb_tst = [], []

for p in tqdm(files, desc="prep"):
    pdf = prep_pdf(p)
    if pdf is None: continue
    N = len(pdf); n_train_core = N - 2*H; n_train_full = N - H
    if n_train_core <= 40: continue

    res_v = build_feature_frames(pdf, n_train_core)
    res_t = build_feature_frames(pdf, n_train_full)
    if res_v is None or res_t is None: continue
    tr_v, te_v, trb_v, teb_v, past_v, fut_v, sc_v = res_v
    tr_t, te_t, trb_t, teb_t, past_t, fut_t, sc_t = res_t

    for ctx in CTX_CHOICES:
        val_items_grid[ctx].append((trb_v, teb_v, past_v, fut_v))
        tst_items_grid[ctx].append((trb_t, teb_t, past_t, fut_t))

    y_val.append(te_v["y"].to_numpy(np.float32))
    y_tst.append(te_t["y"].to_numpy(np.float32))

    y_hist_v = tr_v["y"].to_numpy(np.float32)
    y_hist_t = tr_t["y"].to_numpy(np.float32)
    nb_val.append({"seasonal": seasonal_naive(y_hist_v),
                   "last":     last_value(y_hist_v),
                   "drift":    drift_naive(y_hist_v, k=13)})
    nb_tst.append({"seasonal": seasonal_naive(y_hist_t),
                   "last":     last_value(y_hist_t),
                   "drift":    drift_naive(y_hist_t, k=13)})

    meta.append({"keyword": p.stem, "sc_val": sc_v, "sc_test": sc_t})

n_series = len(meta)
if n_series == 0:
    raise RuntimeError("No eligible series.")

# ---------------- Run Moirai per context, pick best ctx on validation ----------------
def run_ctx(items_grid, label):
    preds = {}
    for ctx in CTX_CHOICES:
        buf = []
        items = items_grid[ctx]
        for i in tqdm(range(0, len(items), BATCH_SERIES), desc=f"{label} ctx={ctx}"):
            chunk = items[i:i+BATCH_SERIES]
            out = moirai_predict_batch(chunk, ctx, num_samples=NUM_SAMPLES)
            buf.extend(out)
        preds[ctx] = np.stack(buf, axis=0)  # z-space
    return preds

print("Predicting validation grid...")
zhat_val = run_ctx(val_items_grid, "val")
print("Predicting test grid...")
zhat_tst = run_ctx(tst_items_grid, "test")

def invert_grid(zhat_by_ctx, which):
    Y = {ctx: [] for ctx in CTX_CHOICES}
    for i in range(n_series):
        med = meta[i]["sc_val"]["med"] if which == "val" else meta[i]["sc_test"]["med"]
        iqr = meta[i]["sc_val"]["iqr"] if which == "val" else meta[i]["sc_test"]["iqr"]
        for ctx in CTX_CHOICES:
            yhat = np.expm1(zhat_by_ctx[ctx][i]*iqr + med) - 1e-6
            Y[ctx].append(np.maximum(yhat, 0.0).astype(np.float32))
    for ctx in CTX_CHOICES:
        Y[ctx] = np.stack(Y[ctx], axis=0)
    return Y

yhat_val = invert_grid(zhat_val, "val")
yhat_tst = invert_grid(zhat_tst, "test")
Yv = np.stack(y_val, axis=0); Yt = np.stack(y_tst, axis=0)

# sMAPE per ctx on validation, choose best ctx per series
val_err_ctx = {ctx: smape_vec(Yv, yhat_val[ctx]) for ctx in CTX_CHOICES}
best_ctx_idx = np.argmin(np.stack([val_err_ctx[c] for c in CTX_CHOICES], axis=1), axis=1)
ctx_arr = np.array(CTX_CHOICES, dtype=np.int32)

moirai_val_best = np.take_along_axis(np.stack([yhat_val[c] for c in CTX_CHOICES], axis=2),
                                     best_ctx_idx[:,None,None], axis=2)[:,:,0]
moirai_tst_best = np.take_along_axis(np.stack([yhat_tst[c] for c in CTX_CHOICES], axis=2),
                                     best_ctx_idx[:,None,None], axis=2)[:,:,0]
moirai_val_err  = smape_vec(Yv, moirai_val_best)

# ---------------- Build gated convex blend from validation ----------------
def inv_error_weights(errs):
    w = (np.maximum(errs, 0.0) + EPS_ERR) ** (-WEIGHT_P)
    w = np.maximum(w, 1e-12); return w / w.sum()

# components: [Moirai_best, seasonal, last, drift]
W = np.zeros((n_series, 4), dtype=np.float32)
use_blend = np.zeros(n_series, dtype=bool)

for i in range(n_series):
    comps_val = np.stack([
        moirai_val_best[i],
        nb_val[i]["seasonal"],
        nb_val[i]["last"],
        nb_val[i]["drift"],
    ], axis=0)

    errs = np.array([smape(Yv[i], comps_val[j]) for j in range(4)], dtype=np.float32)
    # base weights from inverse error
    w = inv_error_weights(errs)

    # enforce Moirai min weight
    if w[0] < MOIRAI_W_MIN:
        surplus = MOIRAI_W_MIN - w[0]
        w[0] = MOIRAI_W_MIN
        if surplus > 0:
            rest = w[1:].sum()
            if rest > 0:
                w[1:] *= (1 - MOIRAI_W_MIN) / rest
            else:
                w[1:] = 0.0

    # compute blended validation forecast & check improvement
    yhat_val_blend = (w[:,None] * comps_val).sum(axis=0)
    s_val_blend = smape(Yv[i], yhat_val_blend)

    if (moirai_val_err[i] - s_val_blend) >= DELTA_MIN:
        use_blend[i] = True
        W[i,:] = w
    else:
        use_blend[i] = False
        W[i,:] = np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float32)  # fallback to Moirai only

# ---------------- Build TEST forecasts accordingly ----------------
Y_pred = np.zeros_like(moirai_tst_best)
for i in range(n_series):
    comps_tst = np.stack([
        moirai_tst_best[i],
        nb_tst[i]["seasonal"],
        nb_tst[i]["last"],
        nb_tst[i]["drift"],
    ], axis=0)
    yhat = (W[i][:,None] * comps_tst).sum(axis=0)
    if USE_CLIP:
        env = np.concatenate([nb_tst[i]["seasonal"], nb_tst[i]["last"], nb_tst[i]["drift"]]).astype(np.float32)
        lo, hi = np.percentile(env, [1,99])
        yhat = np.clip(yhat, lo, hi)
    Y_pred[i] = yhat.astype(np.float32)

# ---------------- Evaluate & save ----------------
s_meta = smape_vec(Yt, Y_pred)
s_moir = smape_vec(Yt, moirai_tst_best)
r_meta = np.sqrt(((Yt - Y_pred)**2).mean(axis=1)).astype(np.float32)

def shares(x):
    return {f"sMAPE≤{t}": float((x <= t).mean()) for t in [10,15,20,25,30,40]}

print("\n=== TEST — Gated blend vs Moirai (best ctx per series) ===")
print("Gated blend:", {"mean": float(s_meta.mean()),
                      "median": float(np.median(s_meta)),
                      **shares(s_meta)})
print("Moirai only:", {"mean": float(s_moir.mean()),
                      "median": float(np.median(s_moir)),
                      **shares(s_moir)})
print(f"Blend accepted on {int(use_blend.sum())}/{n_series} series ({use_blend.mean():.1%}); "
      f"Moirai w_min={MOIRAI_W_MIN}, Δ*={DELTA_MIN}")

weights_df = pd.DataFrame({
    "keyword": [m["keyword"] for m in meta],
    "ctx_picked": ctx_arr[best_ctx_idx],
    "use_blend": use_blend.astype(int),
    "w_moirai": W[:,0], "w_seasonal": W[:,1], "w_last": W[:,2], "w_drift": W[:,3],
})
metrics_df = pd.DataFrame({
    "keyword": [m["keyword"] for m in meta],
    "sMAPE_test_gated": s_meta,
    "sMAPE_test_moirai": s_moir,
    "RMSE_test_gated": r_meta,
})
for t in [10,15,20,25,30,40]:
    metrics_df[f"hit_gated_le_{t}"] = (s_meta <= t).astype(int)

weights_df.to_csv(OUT_WEIGHTS, index=False)
metrics_df.to_csv(OUT_METRICS, index=False)
print(f"\n✅ Saved weights → {OUT_WEIGHTS}")
print(f"✅ Saved metrics → {OUT_METRICS}")

if device == "cuda":
    try:
        peak = torch.cuda.max_memory_allocated() / (1024**3)
        print(f"[GPU] Peak VRAM: {peak:.2f} GiB")
    except Exception:
        pass
